# 06 · CUB + MCBM — two partitions, deliberately unequal

**Full CUB (200) and CUB70 are NOT the same experiment.** Keep them apart:

| partition | what it supports | why |
|-----------|------------------|-----|
| **FULL CUB (200)** | recall-gap axis only | attributes vary within a species → matched-pair recall gap has signal; **no part masks** → no occlusion/grounding |
| **CUB70** (first 70 classes) | occlusion + part-level grounding on **real birds**, relabeling | ships **per-part segmentation masks** |

So CUB70 does much more (the FunnyBirds-style causal probe, on real images). Full
CUB is the weak-but-broad confirmation. Sections are hard-partitioned below.

**Status:** CUB MCBM training is WIP — cells read artifacts once produced and
mark `[pending]` otherwise. Method source: `notebooks/04_cub_analysis.ipynb` (data),
`fb_cbm_renderer_swap_v2.ipynb` (occlusion).

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CURATED = Path(os.environ["CURATED_DATA"])            # cluster data root
REPO = Path.cwd().parent                              # run from curated/notebooks
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"

def need(p, how):
    p = Path(p)
    if not p.exists():
        print(f"[pending] {p}\n  produce it:  {how}")
    return p.exists()


## PART A · FULL CUB (200) — recall-gap axis
Matched-pair recall gap: for a concept the species has, compare model firing on
images where the part is annotated present vs the model's confidence — the
broad, mask-free signal. (Weak on its own; corroborates CUB70.)

In [ ]:
SEED=1
ev = CURATED/"eval"/f"cub-mcbm-s{SEED}.parquet"
if need(ev, "train full-CUB mcbm + build eval table (see RUNBOOK C3/C5)"):
    E = pd.read_parquet(ev)
    # recall gap = mean prob when gt present vs absent, per part
    g = (E.groupby(["part","gt_label"]).prob.mean().unstack(fill_value=np.nan))
    g["recall_gap"] = g.get(1) - g.get(0); display(g.round(3).sort_values("recall_gap"))
    fig,ax=plt.subplots(figsize=(7,3.2))
    gg=g.dropna(subset=["recall_gap"]).sort_values("recall_gap")
    ax.barh(gg.index.astype(str), gg.recall_gap, color=MCBM_C); ax.set_xlabel("recall gap  (P|present − P|absent)")
    ax.set_title("Full CUB · MCBM · per-part recall gap")

## PART B · CUB70 — occlusion grounding on REAL birds  *(the strong test)*
Using per-part segmentation masks: occlude a part and read the retained concept
prob — the CUB analogue of the FunnyBirds deletion test, on real images.
`backwash = retained P of an occluded part`.

In [ ]:
go = CURATED/"grounding"/f"cub70-mcbm-original-s{SEED}.parquet"
if need(go, "train CUB70 mcbm + run CUB70 occlusion probe (RUNBOOK C5)"):
    G = pd.read_parquet(go)
    per = (G.assign(backwash=1-(G.p_intact-G.p_removed)/G.p_intact)
             .groupby("part").backwash.mean().sort_values(ascending=False))
    display(per.round(3))
    fig,ax=plt.subplots(figsize=(6,3.2)); ax.bar(per.index, per.values, color=MCBM_C)
    ax.set_ylim(0,1); ax.set_ylabel("backwash (retained P of occluded part)")
    ax.set_title("CUB70 · MCBM · per-part occlusion backwash"); plt.xticks(rotation=30,ha="right")

### B2 · CUB70 relabeled vs original — does cleaning labels reduce backwash?
The label-standardization critique (STORY §3b): CUB concepts are majority-voted to
class level, so a bird labeled with a part it doesn't visibly show trains backwash.
Relabel per **actual visibility** (masks) and re-measure. `original` vs `relabeled`.

In [ ]:
SEED=1; rows=[]
for lab in ("original","relabeled"):
    p = CURATED/"grounding"/f"cub70-mcbm-{lab}-s{SEED}.parquet"
    if p.exists():
        G=pd.read_parquet(p); rows.append(dict(labels=lab,
            backwash=float(1-(G.p_intact.mean()-G.p_removed.mean())/G.p_intact.mean())))
if len(rows)==2:
    D=pd.DataFrame(rows); display(D.round(3))
    fig,ax=plt.subplots(figsize=(4,3.2)); ax.bar(D.labels,D.backwash,color=[MCBM_C,"#009E73"])
    ax.set_ylim(0,1); ax.set_ylabel("overall backwash"); ax.set_title("CUB70 · MCBM · label cleaning (C6)")
    print("Drop from original→relabeled = backwash attributable to class-level labeling.")
else:
    print("[pending] need BOTH original and relabeled CUB70 MCBM runs (RUNBOOK C6).")

### B3 · CUB70 backwash vs γ (MCBM)
Same minimality question as FunnyBirds, on real birds: does the IB fix grounding
where masks let us measure it? Reads `cub70` rows of a backwash-vs-γ collect.

In [ ]:
bw = CURATED/"cub70_backwash_vs_gamma.csv"
if need(bw, "train CUB70 MCBM sweep + collect (RUNBOOK C5, cub70 variant)"):
    T=pd.read_csv(bw); mc=T[T.model=="mcbm"];
    g=mc.groupby("gamma").backwash.agg(["mean","std"]).reset_index()
    fig,ax=plt.subplots(figsize=(6,3.6))
    ax.errorbar(g.gamma.replace(0,0.02),g["mean"],yerr=g["std"].fillna(0),marker="o",color=MCBM_C)
    ax.set_xscale("log"); ax.set_xlabel("γ"); ax.set_ylabel("CUB70 backwash"); ax.set_ylim(0,1)
    ax.set_title("CUB70 · MCBM · backwash vs γ")

**Takeaway.** CUB70 carries the causal weight (occlusion on real birds +
relabeling); full CUB only corroborates via the recall gap. Read backwash here
against the label-standardization critique — if relabeling drops it, the concepts
were partly reading class-level labels, not parts.